---
title: 3. 用现代 Keras 建立 CSTR 灰箱模型
---

# 3. 用现代 Keras 建立 CSTR 灰箱模型

本 Notebook 用一个等温、恒容、一级不可逆反应 CSTR，比较机理模型、纯神经网络和灰箱模型。数据由明确的合成方程生成，不代表真实工厂数据。

灰箱模型保留稳态物料衡算，让网络只修正速率常数。所有代码使用 Keras 3 的公开 `keras` API。

## 3.1 生成可复现的合成数据

设输入特征顺序为入口浓度 \(C_{A0}\)、停留时间 \(\tau\)、温度 \(T\)。合成数据使用一个 Arrhenius 速率，并叠加小幅度、未纳入名义机理式的速率变化，模拟结构误差。

In [ ]:
import os

# Keras 在 import 时选择后端，所以环境变量必须写在 import keras 之前。
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
import matplotlib.pyplot as plt
import numpy as np
from keras import layers, ops


# 统一用 SI 形式的 R 和能量单位 J/mol，避免 Arrhenius 指数出现单位错误。
R = 8.314  # J / (mol K)
T_REF = 350.0  # K


def make_data(n, temperature_range, seed):
    """生成 CSTR 操作条件和带测量噪声的出口 A 浓度。"""
    # 局部随机数生成器使同一个 seed 可复现，又不会扰动 Notebook 其他随机操作。
    rng = np.random.default_rng(seed)
    c_a0 = rng.uniform(0.5, 2.0, size=n)  # mol / L
    tau = rng.uniform(0.5, 6.0, size=n)  # min
    temperature = rng.uniform(*temperature_range, size=n)  # K

    # 隐藏真值：活化能和弱温度修正均不完全等于后续名义机理模型。
    k_true = 0.035 * np.exp(
        -52_000 / R * (1 / temperature - 1 / T_REF)
    )
    k_true *= 1 + 0.12 * np.sin((temperature - T_REF) / 12)
    # 稳态物料衡算的解析解。例：k*tau=0.5 时，C_A 是 C_A0 的 1/(1+0.5)=2/3。
    c_a = c_a0 / (1 + k_true * tau)
    c_a += rng.normal(0, 0.01, size=n)  # 模拟测量噪声

    features = np.column_stack([c_a0, tau, temperature]).astype("float32")
    target = np.maximum(c_a, 0).astype("float32")[:, None]
    return features, target


# 训练/测试内插使用相同温度区间；额外集从 375 K 开始，完全在训练温区之外。
# 单独留出这个区间，才能看出模型是在学规律还是只记住训练范围。
x_train, y_train = make_data(2_000, (330, 370), seed=1)
x_test, y_test = make_data(500, (330, 370), seed=2)
x_extra, y_extra = make_data(500, (375, 390), seed=3)

print(f"训练样本：{len(x_train)}；内插测试：{len(x_test)}；高温外推：{len(x_extra)}")
print("特征列：[入口浓度 mol/L, 停留时间 min, 温度 K]")

## 3.2 机理模型与灰箱模型

名义机理模型使用正确的 CSTR 衡算式，但活化能固定为一个有误差的先验值；参考速率常数则可训练。灰箱模型在同一机理速率上增加小型网络，输出对速率常数的对数修正 \(\delta\)。

$$
k_{\mathrm{eff}}(x)=k_{\mathrm{Arrhenius}}(T)\exp(\delta_\phi(x)),
\qquad C_A=\frac{C_{A0}}{1+k_{\mathrm{eff}}\tau}.
$$

指数修正保证速率为正。网络最后一层从零初始化，所以训练开始时灰箱模型就是名义机理模型。

In [ ]:
class CSTRModel(keras.Model):
    """保留 CSTR 物料衡算，可选地学习速率常数的工况修正。"""

    def __init__(self, feature_mean, feature_scale, learn_correction):
        super().__init__()
        self.learn_correction = learn_correction
        self.feature_mean = tuple(float(value) for value in feature_mean)
        self.feature_scale = tuple(float(value) for value in feature_scale)

        # 用对数参数化：k_ref=exp(log_k_ref)>0，无需训练后再把负值裁成 0。
        # 初始 k_ref=0.03 min^-1；参考温度 350 K 时 Arrhenius 指数恰为 0。
        self.log_k_ref = self.add_weight(
            name="log_k_ref",
            shape=(),
            initializer=keras.initializers.Constant(np.log(0.03)),
            trainable=True,
        )

        if learn_correction:
            # 末层零初始化令初始 log_correction=0，因此 exp(0)=1，灰箱初始等于机理基线。
            self.correction_net = keras.Sequential(
                [
                    layers.Dense(16, activation="tanh"),
                    layers.Dense(
                        1,
                        kernel_initializer="zeros",
                        bias_initializer="zeros",
                    ),
                ],
                name="rate_correction",
            )

    def call(self, inputs):
        c_a0 = inputs[:, 0:1]
        tau = inputs[:, 1:2]
        temperature = inputs[:, 2:3]

        # Arrhenius 关系以参考温度处的速率常数为参数；T=350 K 时 k_arrhenius=exp(log_k_ref)。
        activation_energy = 42_000.0  # J / mol：故意设置为不准确的名义值
        k_arrhenius = ops.exp(
            self.log_k_ref
            - activation_energy / R * (1 / temperature - 1 / T_REF)
        )

        if self.learn_correction:
            mean = ops.convert_to_tensor(self.feature_mean, dtype=inputs.dtype)
            scale = ops.convert_to_tensor(self.feature_scale, dtype=inputs.dtype)
            # 按训练集均值/标准差缩放输入；例如特征高于均值 1 个标准差时缩放值为 1。
            normalized_inputs = (inputs - mean) / scale
            log_correction = self.correction_net(normalized_inputs)

            # 轻微惩罚修正幅度，鼓励模型优先使用机理解释数据；delta=0.1 相当于速率乘 1.105。
            self.add_loss(1e-3 * ops.mean(ops.square(log_correction)))
        else:
            log_correction = ops.zeros_like(k_arrhenius)

        k_eff = k_arrhenius * ops.exp(log_correction)
        return c_a0 / (1 + k_eff * tau)


# 统计量只从训练特征计算；不能把测试集也用于归一化，否则测试信息进入训练流程。
feature_mean = x_train.mean(axis=0)
feature_scale = x_train.std(axis=0)

physics_model = CSTRModel(feature_mean, feature_scale, learn_correction=False)
greybox_model = CSTRModel(feature_mean, feature_scale, learn_correction=True)

## 3.3 纯神经网络基线

纯神经网络直接从三个操作变量预测出口浓度。`Normalization.adapt` 只读取训练集的特征，避免测试数据参与预处理统计量的估计。

In [ ]:
# 黑箱网络也先标准化输入。adapt 只“拟合缩放统计量”，并不是训练预测网络。
normalizer = layers.Normalization()
normalizer.adapt(x_train)

blackbox_model = keras.Sequential(
    [
        layers.Input(shape=(3,)),
        normalizer,
        # 每层 32 个隐藏单元；最终 Dense(1) 输出一个浓度预测值。
        layers.Dense(32, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1),
    ],
    name="blackbox_cstr",
)

models = {
    "机理模型": physics_model,
    "纯神经网络": blackbox_model,
    "灰箱模型": greybox_model,
}

# 对三种模型用同一个优化器、损失和 MAE 指标，比较时不额外改变训练目标。
for model in models.values():
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
    )

## 3.4 训练并比较误差

使用训练集内部的一部分数据做早停验证；测试集只用于最终比较。高温外推集完全位于训练温度范围之外，因此可以观察模型在新温区的行为，但不能替代真实工厂验证。

In [ ]:
# 固定初始化和批次打乱的随机性，减少模型之间比较时的偶然差异。
keras.utils.set_random_seed(42)
early_stopping = keras.callbacks.EarlyStopping(
    # 验证损失连续 25 轮不改善就停止，并恢复最佳轮次权重，避免只取最后一轮。
    monitor="val_loss",
    patience=25,
    restore_best_weights=True,
)

for name, model in models.items():
    history = model.fit(
        x_train,
        y_train,
        validation_split=0.2,
        epochs=300,
        batch_size=32,
        callbacks=[early_stopping],
        verbose=0,
    )
    print(f"{name}：训练 {len(history.history['loss'])} 个 epoch")

In [ ]:
def regression_metrics(model, features, target):
    """在给定测试工况上计算 RMSE 和 MAE。"""
    prediction = model.predict(features, verbose=0).reshape(-1)
    # 残差定义为预测-观测；平方后正负误差不抵消，开根号恢复 mol/L 单位。
    residual = prediction - target.reshape(-1)
    rmse = np.sqrt(np.mean(residual**2))
    mae = np.mean(np.abs(residual))
    return rmse, mae, prediction


predictions = {}
for split_name, features, target in [
    ("训练温区内插", x_test, y_test),
    ("高温外推", x_extra, y_extra),
]:
    print(f"\n{split_name}")
    predictions[split_name] = {}
    for name, model in models.items():
        rmse, mae, prediction = regression_metrics(model, features, target)
        predictions[split_name][name] = prediction
        print(f"{name:8s} RMSE={rmse:.4f} mol/L, MAE={mae:.4f} mol/L")

## 3.5 画图并解释结果

理想预测落在对角线上。请比较两张图的偏差，并回头检查每个测试温度是否在训练范围内。不要预设灰箱模型一定胜出：训练数据、噪声、模型容量和先验偏差都会影响结果。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

for ax, split_name, target in [
    (axes[0], "训练温区内插", y_test),
    (axes[1], "高温外推", y_extra),
]:
    plot_labels = {
        "机理模型": "Mechanistic",
        "纯神经网络": "Black-box NN",
        "灰箱模型": "Grey-box",
    }
    for name, prediction in predictions[split_name].items():
        ax.scatter(target, prediction, s=12, alpha=0.45, label=plot_labels[name])
    # 统一坐标范围，避免两幅图各自缩放造成“视觉上都很准”的错觉。
    low = min(target.min(), *(p.min() for p in predictions[split_name].values()))
    high = max(target.max(), *(p.max() for p in predictions[split_name].values()))
    title = (
        "Interpolation"
        if split_name == "训练温区内插"
        else "Temperature extrapolation"
    )
    ax.plot([low, high], [low, high], "k--", label="Ideal prediction")
    # 英文图例不依赖运行环境是否安装中文字体。
    ax.set(
        title=title,
        xlabel="Measured outlet concentration (mol/L)",
        ylabel="Predicted outlet concentration (mol/L)",
    )
    ax.legend()

plt.tight_layout()
plt.show()

learned_k_ref = float(ops.convert_to_numpy(ops.exp(physics_model.log_k_ref)))
print(f"机理模型拟合的参考速率常数：{learned_k_ref:.4f} min⁻¹")

## 讨论

- 机理模型的公式正确，但名义活化能错误；它在哪些工况下出现系统偏差？
- 纯网络的训练数据覆盖到多少温度？外推图反映了什么风险？
- 灰箱网络修正的是速率常数，而不是任意出口浓度。这样保留了哪些物理性质？
- 这份合成数据与真实实验之间还缺少哪些不确定性，例如催化剂失活、热效应、停留时间分布和测量偏差？

**边界提醒：** 本模型是假设等温、稳态、完全混合和一级反应。若系统存在显著放热、体积变化、传质限制或动态效应，应扩充机理方程并用独立实验验证。